In [22]:
import os
import openai
from qdrant_client import QdrantClient
from langsmith import Client

from langchain_openai import ChatOpenAI, OpenAIEmbeddings

### Download an example reference data point from LangSmith

In [23]:
from dotenv import load_dotenv

load_dotenv("../../.env")

True

In [24]:
ls_client = Client()

In [25]:
dataset = ls_client.read_dataset(
    dataset_name="coordinator-delegation-evaluation"
)

In [26]:
dataset

Dataset(name='coordinator-delegation-evaluation', description='', data_type=<DataType.kv: 'kv'>, id=UUID('8595d260-4c36-499a-96ed-f33b8e3b0954'), created_at=datetime.datetime(2026, 8, 4, 1, 34, 32, 283720, tzinfo=TzInfo(0)), modified_at=datetime.datetime(2026, 8, 4, 1, 34, 32, 283720, tzinfo=TzInfo(0)), example_count=3, session_count=0, last_session_start_time=None, inputs_schema=None, outputs_schema=None, transformations=None, metadata=None)

In [27]:
reference_inputs = [item.inputs["input"] for item in list(ls_client.list_examples(dataset_id=dataset.id, limit=50))]

In [28]:
reference_inputs

[{'answer': 'I found outdoor speakers, but I didn’t find a clearly green one in the available products.\n\n- If color matters, you may need to refine the request to a specific green shade or brand.\n- For outdoor use, these available products look suitable:\n  - IPX7 waterproof portable speaker with 30W sound\n  - IPX6 waterproof 60W speaker with 24H playtime\n  - IPX6 waterproof 80W speaker with handle and outdoor-focused design\n\nIf you want, I can look for a green speaker in a different style or size.'},
 {'messages': [{'type': 'human',
    'content': 'Can i get some earphones and a laptop?',
    'additional_kwargs': {},
    'response_metadata': {}},
   {'id': 'resp_01d908a14214913a006a713f663c0481999024991528014f21',
    'type': 'ai',
    'content': [{'id': 'rs_01d908a14214913a006a713f66c0cc8199a268378cce82972c',
      'type': 'reasoning',
      'content': [],
      'summary': [],
      'encrypted_content': 'gAAAAABqcT9nLrLnn5VG3ri-S8bzbIZ0yB35BQMhBmSDwAviMtjB40TYLeKHS5WWExOaClHBt

In [29]:
reference_outputs = [item.outputs for item in list(ls_client.list_examples(dataset_id=dataset.id, limit=50))]

In [30]:
reference_outputs

[{'answer': '',
  'coordinator_agent': {'plan': None, 'next_agent': '', 'final_answer': True}},
 {'answer': '',
  'coordinator_agent': {'plan': None, 'next_agent': '', 'final_answer': True}},
 {'answer': '',
  'coordinator_agent': {'plan': [{'task': 'Reserve the best-reviewed items already selected for the user’s cart: earphones product ID B0C13M8GJZ and laptop product ID B0B232NZJL. Confirm whether both items can be reserved in warehouse stock, and report any availability issues if reservation fails.',
     'agent': 'warehouse_manager_agent'}],
   'next_agent': 'warehouse_manager_agent',
   'final_answer': False}}]

In [31]:
reference_inputs = [item.inputs["input"] for item in list(ls_client.list_examples(dataset_id=dataset.id, limit=50))]

In [32]:
reference_inputs

[{'answer': 'I found outdoor speakers, but I didn’t find a clearly green one in the available products.\n\n- If color matters, you may need to refine the request to a specific green shade or brand.\n- For outdoor use, these available products look suitable:\n  - IPX7 waterproof portable speaker with 30W sound\n  - IPX6 waterproof 60W speaker with 24H playtime\n  - IPX6 waterproof 80W speaker with handle and outdoor-focused design\n\nIf you want, I can look for a green speaker in a different style or size.'},
 {'messages': [{'type': 'human',
    'content': 'Can i get some earphones and a laptop?',
    'additional_kwargs': {},
    'response_metadata': {}},
   {'id': 'resp_01d908a14214913a006a713f663c0481999024991528014f21',
    'type': 'ai',
    'content': [{'id': 'rs_01d908a14214913a006a713f66c0cc8199a268378cce82972c',
      'type': 'reasoning',
      'content': [],
      'summary': [],
      'encrypted_content': 'gAAAAABqcT9nLrLnn5VG3ri-S8bzbIZ0yB35BQMhBmSDwAviMtjB40TYLeKHS5WWExOaClHBt

### Coordinator Agent Evaluation

In [33]:
from pydantic import BaseModel

from langchain_openai import ChatOpenAI

from langsmith import traceable

from langchain_core.messages import SystemMessage, AIMessage
from IPython.display import display

from typing import Any, Annotated, List
from pydantic import Field
from operator import add

from jinja2 import Template

In [34]:
class Delegation(BaseModel):
    agent: str = Field(description="The agent to delegate the task to.")
    task: str = Field(description="The task to be performed by the agent.")

class Plan(BaseModel):
    next_agent: str = Field(description="The next agent to invoke")
    plan: List[Delegation] = Field(description="A list of delegations to agents with tasks to be performed in sequence.")

class FinalAgentResponse(BaseModel):

    answer: str = Field(description="Answer to the question")

class CoordinatorAgentProperties(BaseModel):
    iteration: int = 0
    final_answer: bool = False
    plan: List[Delegation] = []
    next_agent: str = ""
    
class State(BaseModel):
    messages: Annotated[List[Any], add] = []
    coordinator_agent: CoordinatorAgentProperties = CoordinatorAgentProperties()
    answer: str = ""

In [35]:
@traceable(
    name="coordinator_agent",
    run_type="llm",
    metadata={
        "ls_provider": "openai",
        "ls_model_name": "gpt-5.4-mini"
    }
)
def coordinator_agent(state) -> dict:
    
    prompt_template = """You are a Coordinator Agent as part of a shopping assistant.

## Instructions

- Your role is to create plans for solving user queries and delegate the tasks accordingly.
- You will be given a conversation history, your task is to create a plan for solving the user's query.
- After the plan is created, you should output the next agent to invoke and the task to be performed by that agent.
- Once an agent finishes its task, you will be handed the control back, you should then review the conversation history and revise the plan.
- If there is a sequence of tasks to be performed by a single agent, you should combine them into a single task.
- Do not route to any agent if the user's query needs clarification or is irelevant. Do it yourself.

## Available Agents

- product_qna_agent: The user is asking a question about a product. This can be a question about available products, their specifications, user reviews etc.
- shopping_cart_agent: The user is asking to add or remove items from the shopping cart or questions about the current shopping cart.
- warehouse_manager_agent: The user is asking to reserve items from the warehouses or about availability of the items in warehouses.

## Examples

Question: "Do you have running shoes under $100?"
Next agent: product_qna_agent

Question: "Can you list the items in my cart?"
Next agent: shopping_cart_agent

Question: "Can you reserve my shopping cart?"
Next agent: warehouse_manager_agent
"""

    template = Template(prompt_template)

    prompt = template.render()

    llm = ChatOpenAI(
        model="gpt-5.4-mini",
        reasoning_effort="low",
        use_responses_api=True
    )
    llm_with_tools = llm.bind_tools(
        [FinalAgentResponse, Plan],
        tool_choice="required"
    )

    response = llm_with_tools.invoke(
        [
            SystemMessage(content=prompt),
            *state.messages
        ]
    )

    final_answer = False
    answer = ""
    plan = []
    next_agent = ""

    def sanitise_response(response):

        for tool_call in response.tool_calls:
            if tool_call.get("name") == "FinalAgentResponse":
                answer = tool_call.get("args").get("answer")

        return AIMessage(content=answer)

    if len(response.tool_calls) > 0:
        if response.tool_calls[0].get("name") == "Plan":
            plan = response.tool_calls[0].get("args").get("plan")
            next_agent = response.tool_calls[0].get("args").get("next_agent")
            response = None
        else:
            for tool_call in response.tool_calls:
                if tool_call.get("name") == "FinalAgentResponse":
                    final_answer = True
                    answer = tool_call.get("args").get("answer")

                    response = sanitise_response(response)

    return {
        "messages": [response] if response else [],
        "coordinator_agent": {
            "iteration": state.coordinator_agent.iteration + 1,
            "final_answer": final_answer,
            "plan": plan,
            "next_agent": next_agent
        },
        "answer": answer
    }

In [38]:
output = coordinator_agent(
    State(
        messages=reference_inputs[1]["messages"],
        coordinator_agent=CoordinatorAgentProperties(
            iteration=0,
            final_answer=False,
            plan=[],
            next_agent=""
        ),
        answer=""
    )
)

In [39]:
output

{'messages': [],
 'coordinator_agent': {'iteration': 1,
  'final_answer': False,
  'plan': [{'agent': 'product_qna_agent',
    'task': 'Find a few suitable earphones and laptop options for the user, then summarize the best matches with key specs and ratings.'}],
  'next_agent': 'product_qna_agent'},
 'answer': ''}

In [40]:
def evaluate_coordinator_delegation(run, example):

    final_answer_match = run["coordinator_agent"]["final_answer"] == example["coordinator_agent"]["final_answer"]
    next_agent_match = run["coordinator_agent"]["next_agent"] == example["coordinator_agent"]["next_agent"]

    return final_answer_match and next_agent_match

In [41]:
evaluate_coordinator_delegation(output, reference_outputs[0])

False

### Run against LangSmith

In [42]:
def evaluate_coordinator_delegation(run, example):

    final_answer_match = run.outputs["coordinator_agent"]["final_answer"] == example.outputs["coordinator_agent"]["final_answer"]
    next_agent_match = run.outputs["coordinator_agent"]["next_agent"] == example.outputs["coordinator_agent"]["next_agent"]

    return final_answer_match and next_agent_match

In [48]:
results = ls_client.evaluate(
    lambda x: coordinator_agent(
        State(
            messages=x["input"]["messages"],
            coordinator_agent=CoordinatorAgentProperties(
                iteration=0,
                final_answer=False,
                plan=[],
                next_agent=""
            ),
            answer=""
        )
    ),
    data="coordinator-delegation-evaluation",
    evaluators=[
        evaluate_coordinator_delegation
    ],
    experiment_prefix="coordinator-delegation",
    max_concurrency=10,
)

View the evaluation results for experiment: 'coordinator-delegation-f028a54b' at:
https://smith.langchain.com/o/b99d0ed7-3848-4b7d-a09d-6fa565e05a8b/datasets/8595d260-4c36-499a-96ed-f33b8e3b0954/compare?selectedSessions=323559bd-27be-46d2-8a6e-3161f396b026




0it [00:00, ?it/s]Error running target function: 'messages'
Traceback (most recent call last):
  File "/Users/tk/Projects/chatbot_poc_v2/.venv/lib/python3.12/site-packages/langsmith/evaluation/_runner.py", line 1976, in _forward
    fn(*args, langsmith_extra=langsmith_extra)
  File "/var/folders/x9/1y3l4mfd6hbgv0p0j9p7h76m0000gn/T/ipykernel_9153/55481811.py", line 4, in <lambda>
    messages=x["input"]["messages"],
             ~~~~~~~~~~^^^^^^^^^^^^
KeyError: 'messages'
Error running evaluator <DynamicRunEvaluator evaluate_coordinator_delegation> on run 019fce75-f40e-74b0-8a8c-6a19b66daeaf: KeyError('coordinator_agent')
Traceback (most recent call last):
  File "/Users/tk/Projects/chatbot_poc_v2/.venv/lib/python3.12/site-packages/langsmith/evaluation/_runner.py", line 1672, in _run_evaluators
    evaluator_response = evaluator.evaluate_run(  # type: ignore[call-arg]
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/tk/Projects/chatbot_poc_v2/.ve

### Coordinator Agent V1

In [49]:
class Delegation(BaseModel):
    agent: str = Field(description="The agent to delegate the task to.")
    task: str = Field(description="The task to be performed by the agent.")

class Plan(BaseModel):
    next_agent: str = Field(description="The next agent to invoke")
    plan: List[Delegation] = Field(description="A list of delegations to agents with tasks to be performed in sequence.")

class FinalAgentResponse(BaseModel):

    answer: str = Field(description="Answer to the question")

class CoordinatorAgentProperties(BaseModel):
    iteration: int = 0
    final_answer: bool = False
    plan: List[Delegation] = []
    next_agent: str = ""
    
class State(BaseModel):
    messages: Annotated[List[Any], add] = []
    coordinator_agent: CoordinatorAgentProperties = CoordinatorAgentProperties()
    answer: str = ""

In [50]:
@traceable(
    name="coordinator_agent",
    run_type="llm",
    metadata={
        "ls_provider": "openai",
        "ls_model_name": "gpt-5.4-mini"
    }
)
def coordinator_agent(state) -> dict:
    
    prompt_template = """You are a Coordinator Agent as part of a shopping assistant.

## Instructions

- Your role is to delegate work to worker agents in order to solve user queries.
- You should output the next agent to invoke.
- Once an agent finishes its task, you will be handed the control back, you should then review the conversation history and output the next delegation via the `Plan` tool or output the final answer via the `FinalAgentResponse` tool.
- Do not route to any agent if the user's query needs clarification or is irelevant. Do it yourself.
- Only route to the `warehouse_manager_agent` if the user has confirmed that they want to reserve an item and it was already successfully added to the shopping cart.
- Only route to the `shopping_cart_agent` if the user specifically asks about their shopping cart or wants to add or remove items from the shopping cart.
- Do not delegate work to the same agent twice in a row.

## Routing rules

Match the user's latest message to one of these patterns:

1. PRODUCT QUESTION — asks about products, specs, reviews, availability, suggestions.
    Examples: "can I get some earphones", "show me laptops", "what watches do you have", "tell me about X", "is X any good", "I'd like to see some X".
    → Route ONLY to `product_qna_agent`. When it returns, emit `FinalAgentResponse`. Do NOT chain to cart or warehouse.

2. EXPLICIT CART INSTRUCTION — uses cart vocabulary.
    Examples: "add to my cart", "remove from my cart", "what's in my cart", "put X in my cart", "delete X from cart".
    → Route to `shopping_cart_agent`. (If items must be discovered first, route to `product_qna_agent` first.)

3. EXPLICIT RESERVATION INSTRUCTION — uses warehouse/reservation vocabulary.
    Examples: "reserve these", "hold these for me", "check warehouse stock for X".
    → Route to `warehouse_manager_agent` ONLY AFTER items are confirmed in the cart.

Default rule: if the user's message does NOT contain explicit cart or warehouse vocabulary, do NOT delegate to those agents. Phrases like "can I get", "I want",
"I'd like", "show me", "I need" are PRODUCT QUESTIONS, not purchase instructions. The user must explicitly say "add", "cart", "reserve", "buy", or "order" before those agents are in scope.

## Available Agents

- product_qna_agent: The user is asking a question about a product. This can be a question about available products, their specifications, user reviews etc.
- shopping_cart_agent: The user is asking to add or remove items from the shopping cart or questions about the current shopping cart.
- warehouse_manager_agent: The user is asking to reserve items from the warehouses or about availability of the items in warehouses.
"""

    template = Template(prompt_template)

    prompt = template.render()

    llm = ChatOpenAI(
        model="gpt-5.4-mini",
        reasoning_effort="low",
        use_responses_api=True
    )
    llm_with_tools = llm.bind_tools(
        [FinalAgentResponse, Plan],
        tool_choice="required"
    )

    response = llm_with_tools.invoke(
        [
            SystemMessage(content=prompt),
            *state.messages
        ]
    )

    final_answer = False
    answer = ""
    plan = []
    next_agent = ""

    def sanitise_response(response):

        for tool_call in response.tool_calls:
            if tool_call.get("name") == "FinalAgentResponse":
                answer = tool_call.get("args").get("answer")

        return AIMessage(content=answer)

    if len(response.tool_calls) > 0:
        if response.tool_calls[0].get("name") == "Plan":
            plan = response.tool_calls[0].get("args").get("plan")
            next_agent = response.tool_calls[0].get("args").get("next_agent")
            response = None
        else:
            for tool_call in response.tool_calls:
                if tool_call.get("name") == "FinalAgentResponse":
                    final_answer = True
                    answer = tool_call.get("args").get("answer")

                    response = sanitise_response(response)

    return {
        "messages": [response] if response else [],
        "coordinator_agent": {
            "iteration": state.coordinator_agent.iteration + 1,
            "final_answer": final_answer,
            "plan": plan,
            "next_agent": next_agent
        },
        "answer": answer
    }

In [51]:
results = ls_client.evaluate(
    lambda x: coordinator_agent(
        State(
            messages=x["input"]["messages"],
            coordinator_agent=CoordinatorAgentProperties(
                iteration=0,
                final_answer=False,
                plan=[],
                next_agent=""
            ),
            answer=""
        )
    ),
    data="coordinator-delegation-evaluation",
    evaluators=[
        evaluate_coordinator_delegation
    ],
    experiment_prefix="coordinator-delegation",
    max_concurrency=10,
    num_repetitions=5
)

View the evaluation results for experiment: 'coordinator-delegation-fea86619' at:
https://smith.langchain.com/o/b99d0ed7-3848-4b7d-a09d-6fa565e05a8b/datasets/8595d260-4c36-499a-96ed-f33b8e3b0954/compare?selectedSessions=671924f0-b5c5-4ad8-8f8d-13c15dd9b6a2




0it [00:00, ?it/s]Error running target function: 'messages'
Traceback (most recent call last):
  File "/Users/tk/Projects/chatbot_poc_v2/.venv/lib/python3.12/site-packages/langsmith/evaluation/_runner.py", line 1976, in _forward
    fn(*args, langsmith_extra=langsmith_extra)
  File "/var/folders/x9/1y3l4mfd6hbgv0p0j9p7h76m0000gn/T/ipykernel_9153/361648355.py", line 4, in <lambda>
    messages=x["input"]["messages"],
             ~~~~~~~~~~^^^^^^^^^^^^
KeyError: 'messages'
Error running target function: 'messages'
Traceback (most recent call last):
  File "/Users/tk/Projects/chatbot_poc_v2/.venv/lib/python3.12/site-packages/langsmith/evaluation/_runner.py", line 1976, in _forward
    fn(*args, langsmith_extra=langsmith_extra)
  File "/var/folders/x9/1y3l4mfd6hbgv0p0j9p7h76m0000gn/T/ipykernel_9153/361648355.py", line 4, in <lambda>
    messages=x["input"]["messages"],
             ~~~~~~~~~~^^^^^^^^^^^^
KeyError: 'messages'
Error running target function: 'messages'
Traceback (most recent

### Coordinator Agent V2

In [52]:
class Delegation(BaseModel):
    agent: str = Field(description="The agent to delegate the task to.")
    task: str = Field(description="The task to be performed by the agent.")

class Plan(BaseModel):
    next_agent: str = Field(description="The next agent to invoke")
    plan: List[Delegation] = Field(description="A list of delegations to agents with tasks to be performed in sequence.")

class FinalAgentResponse(BaseModel):

    answer: str = Field(description="Answer to the question")

class CoordinatorAgentProperties(BaseModel):
    iteration: int = 0
    final_answer: bool = False
    plan: List[Delegation] = []
    next_agent: str = ""
    
class State(BaseModel):
    messages: Annotated[List[Any], add] = []
    coordinator_agent: CoordinatorAgentProperties = CoordinatorAgentProperties()
    answer: str = ""

In [53]:
@traceable(
    name="coordinator_agent",
    run_type="llm",
    metadata={
        "ls_provider": "openai",
        "ls_model_name": "gpt-5.4-mini"
    }
)
def coordinator_agent(state) -> dict:
    
    prompt_template = """You are a Coordinator Agent as part of a shopping assistant.

## Instructions

- Your role is to delegate work to worker agents in order to solve user queries.
- You should output the next agent to invoke.
- Once an agent finishes its task, you will be handed the control back, you should then review the conversation history and output the next delegation via the `Plan` tool.
- If you do not need to delegate work to any agent, output the final answer via the `FinalAgentResponse` tool.
- Do not route to any agent if the user's query needs clarification or is irelevant. Do it yourself.
- Only route to the `warehouse_manager_agent` if the user has confirmed that they want to reserve an item and it was already successfully added to the shopping cart.
- Only route to the `shopping_cart_agent` if the user specifically asks about their shopping cart or wants to add or remove items from the shopping cart.
- Do not delegate work to the same agent twice in a row.

## Routing rules

Match the user's latest message to one of these patterns:

1. PRODUCT QUESTION — asks about products, specs, reviews, availability, suggestions.
    Examples: "can I get some earphones", "show me laptops", "what watches do you have", "tell me about X", "is X any good", "I'd like to see some X".
    → Route ONLY to `product_qna_agent`. When it returns, emit `FinalAgentResponse`. Do NOT chain to cart or warehouse.

2. EXPLICIT CART INSTRUCTION — uses cart vocabulary.
    Examples: "add to my cart", "remove from my cart", "what's in my cart", "put X in my cart", "delete X from cart".
    → Route to `shopping_cart_agent`. (If items must be discovered first, route to `product_qna_agent` first.)

3. EXPLICIT RESERVATION INSTRUCTION — uses warehouse/reservation vocabulary.
    Examples: "reserve these", "hold these for me", "check warehouse stock for X".
    → Route to `warehouse_manager_agent` ONLY AFTER items are confirmed in the cart.

Default rule: if the user's message does NOT contain explicit cart or warehouse vocabulary, do NOT delegate to those agents. Phrases like "can I get", "I want",
"I'd like", "show me", "I need" are PRODUCT QUESTIONS, not purchase instructions. The user must explicitly say "add", "cart", "reserve", "buy", or "order" before those agents are in scope.

## Available Agents

- product_qna_agent: The user is asking a question about a product. This can be a question about available products, their specifications, user reviews etc.
- shopping_cart_agent: The user is asking to add or remove items from the shopping cart or questions about the current shopping cart.
- warehouse_manager_agent: The user is asking to reserve items from the warehouses or about availability of the items in warehouses.
"""

    template = Template(prompt_template)

    prompt = template.render()

    llm = ChatOpenAI(
        model="gpt-5.4-mini",
        reasoning_effort="low",
        use_responses_api=True
    )
    llm_with_tools = llm.bind_tools(
        [FinalAgentResponse, Plan],
        tool_choice="required"
    )

    response = llm_with_tools.invoke(
        [
            SystemMessage(content=prompt),
            *state.messages
        ]
    )

    final_answer = False
    answer = ""
    plan = []
    next_agent = ""

    def sanitise_response(response):

        for tool_call in response.tool_calls:
            if tool_call.get("name") == "FinalAgentResponse":
                answer = tool_call.get("args").get("answer")

        return AIMessage(content=answer)

    if len(response.tool_calls) > 0:
        if response.tool_calls[0].get("name") == "Plan":
            plan = response.tool_calls[0].get("args").get("plan")
            next_agent = response.tool_calls[0].get("args").get("next_agent")
            response = None
        else:
            for tool_call in response.tool_calls:
                if tool_call.get("name") == "FinalAgentResponse":
                    final_answer = True
                    answer = tool_call.get("args").get("answer")

                    response = sanitise_response(response)

    return {
        "messages": [response] if response else [],
        "coordinator_agent": {
            "iteration": state.coordinator_agent.iteration + 1,
            "final_answer": final_answer,
            "plan": plan,
            "next_agent": next_agent
        },
        "answer": answer
    }

In [54]:
results = ls_client.evaluate(
    lambda x: coordinator_agent(
        State(
            messages=x["input"]["messages"],
            coordinator_agent=CoordinatorAgentProperties(
                iteration=0,
                final_answer=False,
                plan=[],
                next_agent=""
            ),
            answer=""
        )
    ),
    data="coordinator-delegation-evaluation",
    evaluators=[
        evaluate_coordinator_delegation
    ],
    experiment_prefix="coordinator-delegation",
    max_concurrency=10,
    num_repetitions=5
)

View the evaluation results for experiment: 'coordinator-delegation-bfaa932e' at:
https://smith.langchain.com/o/b99d0ed7-3848-4b7d-a09d-6fa565e05a8b/datasets/8595d260-4c36-499a-96ed-f33b8e3b0954/compare?selectedSessions=fe07f92d-4f9e-4c18-aaf1-bba8b61df134




0it [00:00, ?it/s]Error running target function: 'messages'
Traceback (most recent call last):
  File "/Users/tk/Projects/chatbot_poc_v2/.venv/lib/python3.12/site-packages/langsmith/evaluation/_runner.py", line 1976, in _forward
    fn(*args, langsmith_extra=langsmith_extra)
  File "/var/folders/x9/1y3l4mfd6hbgv0p0j9p7h76m0000gn/T/ipykernel_9153/361648355.py", line 4, in <lambda>
    messages=x["input"]["messages"],
             ~~~~~~~~~~^^^^^^^^^^^^
KeyError: 'messages'
Error running target function: 'messages'
Traceback (most recent call last):
  File "/Users/tk/Projects/chatbot_poc_v2/.venv/lib/python3.12/site-packages/langsmith/evaluation/_runner.py", line 1976, in _forward
    fn(*args, langsmith_extra=langsmith_extra)
  File "/var/folders/x9/1y3l4mfd6hbgv0p0j9p7h76m0000gn/T/ipykernel_9153/361648355.py", line 4, in <lambda>
    messages=x["input"]["messages"],
             ~~~~~~~~~~^^^^^^^^^^^^
KeyError: 'messages'
Error running target function: 'messages'
Traceback (most recent

### Coordinator Agent V3

In [55]:
class Delegation(BaseModel):
    agent: str = Field(description="The agent to delegate the task to.")
    task: str = Field(description="The task to be performed by the agent.")

class Plan(BaseModel):
    next_agent: str = Field(description="The next agent to invoke")
    plan: List[Delegation] = Field(description="A list of delegations to agents with tasks to be performed in sequence.")

class FinalAgentResponse(BaseModel):

    answer: str = Field(description="Answer to the question")

class CoordinatorAgentProperties(BaseModel):
    iteration: int = 0
    final_answer: bool = False
    plan: List[Delegation] = []
    next_agent: str = ""
    
class State(BaseModel):
    messages: Annotated[List[Any], add] = []
    coordinator_agent: CoordinatorAgentProperties = CoordinatorAgentProperties()
    answer: str = ""

In [56]:
@traceable(
    name="coordinator_agent",
    run_type="llm",
    metadata={
        "ls_provider": "openai",
        "ls_model_name": "gpt-5.4-mini"
    }
)
def coordinator_agent(state) -> dict:
    
    prompt_template = """You are a Coordinator Agent as part of a shopping assistant.

## Instructions

- Your role is to delegate work to worker agents in order to solve user queries.
- You should output the next agent to invoke.
- Once an agent finishes its task, you will be handed the control back, you should then review the conversation history and output the next delegation via the `Plan` tool.
- If you do not need to delegate work to any agent, output the final answer via the `FinalAgentResponse` tool.
- Do not route to any agent if the user's query needs clarification or is irelevant. Do it yourself.
- Only route to the `warehouse_manager_agent` if the user has confirmed that they want to reserve an item and it was already successfully added to the shopping cart.
- Only route to the `shopping_cart_agent` if the user specifically asks about their shopping cart or wants to add or remove items from the shopping cart.
- Do not delegate work to the same agent twice in a row.

## Routing rules

Match the user's latest message to one of these patterns:

1. PRODUCT QUESTION — asks about products, specs, reviews, availability, suggestions.
    Examples: "can I get some earphones", "show me laptops", "what watches do you have", "tell me about X", "is X any good", "I'd like to see some X".
    → Route ONLY to `product_qna_agent`. When it returns, emit `FinalAgentResponse`. Do NOT chain to cart or warehouse.

2. EXPLICIT CART INSTRUCTION — uses cart vocabulary.
    Examples: "add to my cart", "remove from my cart", "what's in my cart", "put X in my cart", "delete X from cart".
    → Route to `shopping_cart_agent`. (If items must be discovered first, route to `product_qna_agent` first.)

3. EXPLICIT RESERVATION INSTRUCTION — uses warehouse/reservation vocabulary.
    Examples: "reserve these", "hold these for me", "check warehouse stock for X".
    → Route to `warehouse_manager_agent` ONLY AFTER items are confirmed in the cart.

Default rule: if the user's message does NOT contain explicit cart or warehouse vocabulary, do NOT delegate to those agents. Phrases like "can I get", "I want",
"I'd like", "show me", "I need" are PRODUCT QUESTIONS, not purchase instructions. The user must explicitly say "add", "cart", "reserve", "buy", or "order" before those agents are in scope.

## Available Agents

- product_qna_agent: The user is asking a question about a product. This can be a question about available products, their specifications, user reviews etc.
- shopping_cart_agent: The user is asking to add or remove items from the shopping cart or questions about the current shopping cart.
- warehouse_manager_agent: The user is asking to reserve items from the warehouses or about availability of the items in warehouses.
"""

    template = Template(prompt_template)

    prompt = template.render()

    llm = ChatOpenAI(
        model="gpt-5.4-mini",
        reasoning_effort="medium",
        use_responses_api=True
    )
    llm_with_tools = llm.bind_tools(
        [FinalAgentResponse, Plan],
        tool_choice="required"
    )

    response = llm_with_tools.invoke(
        [
            SystemMessage(content=prompt),
            *state.messages
        ]
    )

    final_answer = False
    answer = ""
    plan = []
    next_agent = ""

    def sanitise_response(response):

        for tool_call in response.tool_calls:
            if tool_call.get("name") == "FinalAgentResponse":
                answer = tool_call.get("args").get("answer")

        return AIMessage(content=answer)

    if len(response.tool_calls) > 0:
        if response.tool_calls[0].get("name") == "Plan":
            plan = response.tool_calls[0].get("args").get("plan")
            next_agent = response.tool_calls[0].get("args").get("next_agent")
            response = None
        else:
            for tool_call in response.tool_calls:
                if tool_call.get("name") == "FinalAgentResponse":
                    final_answer = True
                    answer = tool_call.get("args").get("answer")

                    response = sanitise_response(response)

    return {
        "messages": [response] if response else [],
        "coordinator_agent": {
            "iteration": state.coordinator_agent.iteration + 1,
            "final_answer": final_answer,
            "plan": plan,
            "next_agent": next_agent
        },
        "answer": answer
    }

In [57]:
results = ls_client.evaluate(
    lambda x: coordinator_agent(
        State(
            messages=x["input"]["messages"],
            coordinator_agent=CoordinatorAgentProperties(
                iteration=0,
                final_answer=False,
                plan=[],
                next_agent=""
            ),
            answer=""
        )
    ),
    data="coordinator-delegation-evaluation",
    evaluators=[
        evaluate_coordinator_delegation
    ],
    experiment_prefix="coordinator-delegation",
    max_concurrency=10,
    num_repetitions=5
)

View the evaluation results for experiment: 'coordinator-delegation-312bf652' at:
https://smith.langchain.com/o/b99d0ed7-3848-4b7d-a09d-6fa565e05a8b/datasets/8595d260-4c36-499a-96ed-f33b8e3b0954/compare?selectedSessions=314dd5de-37ae-4bbc-aae8-d80c75da0afc




0it [00:00, ?it/s]Error running target function: 'messages'
Traceback (most recent call last):
  File "/Users/tk/Projects/chatbot_poc_v2/.venv/lib/python3.12/site-packages/langsmith/evaluation/_runner.py", line 1976, in _forward
    fn(*args, langsmith_extra=langsmith_extra)
  File "/var/folders/x9/1y3l4mfd6hbgv0p0j9p7h76m0000gn/T/ipykernel_9153/361648355.py", line 4, in <lambda>
    messages=x["input"]["messages"],
             ~~~~~~~~~~^^^^^^^^^^^^
KeyError: 'messages'
Error running target function: 'messages'
Traceback (most recent call last):
  File "/Users/tk/Projects/chatbot_poc_v2/.venv/lib/python3.12/site-packages/langsmith/evaluation/_runner.py", line 1976, in _forward
    fn(*args, langsmith_extra=langsmith_extra)
  File "/var/folders/x9/1y3l4mfd6hbgv0p0j9p7h76m0000gn/T/ipykernel_9153/361648355.py", line 4, in <lambda>
    messages=x["input"]["messages"],
             ~~~~~~~~~~^^^^^^^^^^^^
KeyError: 'messages'
Error running target function: 'messages'
Traceback (most recent

### Coordinator Agent V4

In [58]:
class Delegation(BaseModel):
    agent: str = Field(description="The agent to delegate the task to.")
    task: str = Field(description="The task to be performed by the agent.")

class Plan(BaseModel):
    next_agent: str = Field(description="The next agent to invoke")
    # plan: List[Delegation] = Field(description="A list of delegations to agents with tasks to be performed in sequence.")

class FinalAgentResponse(BaseModel):

    answer: str = Field(description="Answer to the question")

class CoordinatorAgentProperties(BaseModel):
    iteration: int = 0
    final_answer: bool = False
    # plan: List[Delegation] = []
    next_agent: str = ""
    
class State(BaseModel):
    messages: Annotated[List[Any], add] = []
    coordinator_agent: CoordinatorAgentProperties = CoordinatorAgentProperties()
    answer: str = ""

In [59]:
@traceable(
    name="coordinator_agent",
    run_type="llm",
    metadata={
        "ls_provider": "openai",
        "ls_model_name": "gpt-5.4-mini"
    }
)
def coordinator_agent(state) -> dict:
    
    prompt_template = """You are a Coordinator Agent as part of a shopping assistant.

## Instructions

- Your role is to delegate work to worker agents in order to solve user queries.
- You should output the next agent to invoke.
- Once an agent finishes its task, you will be handed the control back, you should then review the conversation history and output the next delegation via the `Plan` tool.
- If you do not need to delegate work to any agent, output the final answer via the `FinalAgentResponse` tool.
- Do not route to any agent if the user's query needs clarification or is irelevant. Do it yourself.
- Only route to the `warehouse_manager_agent` if the user has confirmed that they want to reserve an item and it was already successfully added to the shopping cart.
- Only route to the `shopping_cart_agent` if the user specifically asks about their shopping cart or wants to add or remove items from the shopping cart.
- Do not delegate work to the same agent twice in a row.

## Routing rules

Match the user's latest message to one of these patterns:

1. PRODUCT QUESTION — asks about products, specs, reviews, availability, suggestions.
    Examples: "can I get some earphones", "show me laptops", "what watches do you have", "tell me about X", "is X any good", "I'd like to see some X".
    → Route ONLY to `product_qna_agent`. When it returns, emit `FinalAgentResponse`. Do NOT chain to cart or warehouse.

2. EXPLICIT CART INSTRUCTION — uses cart vocabulary.
    Examples: "add to my cart", "remove from my cart", "what's in my cart", "put X in my cart", "delete X from cart".
    → Route to `shopping_cart_agent`. (If items must be discovered first, route to `product_qna_agent` first.)

3. EXPLICIT RESERVATION INSTRUCTION — uses warehouse/reservation vocabulary.
    Examples: "reserve these", "hold these for me", "check warehouse stock for X".
    → Route to `warehouse_manager_agent` ONLY AFTER items are confirmed in the cart.

Default rule: if the user's message does NOT contain explicit cart or warehouse vocabulary, do NOT delegate to those agents. Phrases like "can I get", "I want",
"I'd like", "show me", "I need" are PRODUCT QUESTIONS, not purchase instructions. The user must explicitly say "add", "cart", "reserve", "buy", or "order" before those agents are in scope.

## Available Agents

- product_qna_agent: The user is asking a question about a product. This can be a question about available products, their specifications, user reviews etc.
- shopping_cart_agent: The user is asking to add or remove items from the shopping cart or questions about the current shopping cart.
- warehouse_manager_agent: The user is asking to reserve items from the warehouses or about availability of the items in warehouses.
"""

    template = Template(prompt_template)

    prompt = template.render()

    llm = ChatOpenAI(
        model="gpt-5.4-mini",
        reasoning_effort="medium",
        use_responses_api=True
    )
    llm_with_tools = llm.bind_tools(
        [FinalAgentResponse, Plan],
        tool_choice="required"
    )

    response = llm_with_tools.invoke(
        [
            SystemMessage(content=prompt),
            *state.messages
        ]
    )

    final_answer = False
    answer = ""
    # plan = []
    next_agent = ""

    def sanitise_response(response):

        for tool_call in response.tool_calls:
            if tool_call.get("name") == "FinalAgentResponse":
                answer = tool_call.get("args").get("answer")

        return AIMessage(content=answer)

    if len(response.tool_calls) > 0:
        if response.tool_calls[0].get("name") == "Plan":
            # plan = response.tool_calls[0].get("args").get("plan")
            next_agent = response.tool_calls[0].get("args").get("next_agent")
            response = None
        else:
            for tool_call in response.tool_calls:
                if tool_call.get("name") == "FinalAgentResponse":
                    final_answer = True
                    answer = tool_call.get("args").get("answer")

                    response = sanitise_response(response)

    return {
        "messages": [response] if response else [],
        "coordinator_agent": {
            "iteration": state.coordinator_agent.iteration + 1,
            "final_answer": final_answer,
            # "plan": plan,
            "next_agent": next_agent
        },
        "answer": answer
    }

In [60]:
results = ls_client.evaluate(
    lambda x: coordinator_agent(
        State(
            messages=x["input"]["messages"],
            coordinator_agent=CoordinatorAgentProperties(
                iteration=0,
                final_answer=False,
                plan=[],
                next_agent=""
            ),
            answer=""
        )
    ),
    data="coordinator-delegation-evaluation",
    evaluators=[
        evaluate_coordinator_delegation
    ],
    experiment_prefix="coordinator-delegation",
    max_concurrency=10,
    num_repetitions=5
)

View the evaluation results for experiment: 'coordinator-delegation-22af56b8' at:
https://smith.langchain.com/o/b99d0ed7-3848-4b7d-a09d-6fa565e05a8b/datasets/8595d260-4c36-499a-96ed-f33b8e3b0954/compare?selectedSessions=b967d307-f1ca-41bf-9f8e-24ed49b76c2e




0it [00:00, ?it/s]Error running target function: 'messages'
Traceback (most recent call last):
  File "/Users/tk/Projects/chatbot_poc_v2/.venv/lib/python3.12/site-packages/langsmith/evaluation/_runner.py", line 1976, in _forward
    fn(*args, langsmith_extra=langsmith_extra)
  File "/var/folders/x9/1y3l4mfd6hbgv0p0j9p7h76m0000gn/T/ipykernel_9153/361648355.py", line 4, in <lambda>
    messages=x["input"]["messages"],
             ~~~~~~~~~~^^^^^^^^^^^^
KeyError: 'messages'
Error running target function: 'messages'
Traceback (most recent call last):
  File "/Users/tk/Projects/chatbot_poc_v2/.venv/lib/python3.12/site-packages/langsmith/evaluation/_runner.py", line 1976, in _forward
    fn(*args, langsmith_extra=langsmith_extra)
  File "/var/folders/x9/1y3l4mfd6hbgv0p0j9p7h76m0000gn/T/ipykernel_9153/361648355.py", line 4, in <lambda>
    messages=x["input"]["messages"],
             ~~~~~~~~~~^^^^^^^^^^^^
KeyError: 'messages'
Error running target function: 'messages'
Traceback (most recent

### Coordinator Agent V5

In [61]:
class Delegation(BaseModel):
    agent: str = Field(description="The agent to delegate the task to.")
    task: str = Field(description="The task to be performed by the agent.")

class Plan(BaseModel):
    next_agent: str = Field(description="The next agent to invoke")
    # plan: List[Delegation] = Field(description="A list of delegations to agents with tasks to be performed in sequence.")

class FinalAgentResponse(BaseModel):

    answer: str = Field(description="Answer to the question")

class CoordinatorAgentProperties(BaseModel):
    iteration: int = 0
    final_answer: bool = False
    # plan: List[Delegation] = []
    next_agent: str = ""
    
class State(BaseModel):
    messages: Annotated[List[Any], add] = []
    coordinator_agent: CoordinatorAgentProperties = CoordinatorAgentProperties()
    answer: str = ""

In [62]:
@traceable(
    name="coordinator_agent",
    run_type="llm",
    metadata={
        "ls_provider": "openai",
        "ls_model_name": "gpt-5.4-mini"
    }
)
def coordinator_agent(state) -> dict:
    
    prompt_template = """You are a Coordinator Agent as part of a shopping assistant.

## Instructions

- Your role is to delegate work to worker agents in order to solve user queries.
- You should output the next agent to invoke.
- Once an agent finishes its task, you will be handed the control back, you should then review the conversation history and output the next delegation via the `Plan` tool.
- If you do not need to delegate work to any agent, output the final answer via the `FinalAgentResponse` tool.
- Do not route to any agent if the user's query needs clarification or is irelevant. Do it yourself.
- Only route to the `warehouse_manager_agent` if the user has confirmed that they want to reserve an item and it was already successfully added to the shopping cart.
- Only route to the `shopping_cart_agent` if the user specifically asks about their shopping cart or wants to add or remove items from the shopping cart.
- Do not delegate work to the same agent twice in a row.

## Routing rules

Match the user's latest message to one of these patterns:

1. PRODUCT QUESTION — asks about products, specs, reviews, availability, suggestions.
    Examples: "can I get some earphones", "show me laptops", "what watches do you have", "tell me about X", "is X any good", "I'd like to see some X".
    → Route ONLY to `product_qna_agent`. When it returns, emit `FinalAgentResponse`. Do NOT chain to cart or warehouse.

2. EXPLICIT CART INSTRUCTION — uses cart vocabulary.
    Examples: "add to my cart", "remove from my cart", "what's in my cart", "put X in my cart", "delete X from cart".
    → Route to `shopping_cart_agent`. (If items must be discovered first, route to `product_qna_agent` first.)

3. EXPLICIT RESERVATION INSTRUCTION — uses warehouse/reservation vocabulary.
    Examples: "reserve these", "hold these for me", "check warehouse stock for X".
    → Route to `warehouse_manager_agent` ONLY AFTER items are confirmed in the cart.

Default rule: if the user's message does NOT contain explicit cart or warehouse vocabulary, do NOT delegate to those agents. Phrases like "can I get", "I want",
"I'd like", "show me", "I need" are PRODUCT QUESTIONS, not purchase instructions. The user must explicitly say "add", "cart", "reserve", "buy", or "order" before those agents are in scope.

## Available Agents

- product_qna_agent: The user is asking a question about a product. This can be a question about available products, their specifications, user reviews etc.
- shopping_cart_agent: The user is asking to add or remove items from the shopping cart or questions about the current shopping cart.
- warehouse_manager_agent: The user is asking to reserve items from the warehouses or about availability of the items in warehouses.
"""

    template = Template(prompt_template)

    prompt = template.render()

    llm = ChatOpenAI(
        model="gpt-5.4-mini",
        reasoning_effort="medium",
        use_responses_api=True
    )
    llm_with_tools = llm.bind_tools(
        [FinalAgentResponse, Plan],
        tool_choice="required"
    )

    response = llm_with_tools.invoke(
        [
            SystemMessage(content=prompt),
            *state.messages
        ]
    )

    final_answer = False
    answer = ""
    # plan = []
    next_agent = ""

    def sanitise_response(response):

        for tool_call in response.tool_calls:
            if tool_call.get("name") == "FinalAgentResponse":
                answer = tool_call.get("args").get("answer")

        return AIMessage(content=answer)

    if len(response.tool_calls) > 0:
        if response.tool_calls[0].get("name") == "Plan":
            # plan = response.tool_calls[0].get("args").get("plan")
            next_agent = response.tool_calls[0].get("args").get("next_agent")
            response = None
        else:
            for tool_call in response.tool_calls:
                if tool_call.get("name") == "FinalAgentResponse":
                    final_answer = True
                    answer = tool_call.get("args").get("answer")

                    response = sanitise_response(response)

    return {
        "messages": [response] if response else [],
        "coordinator_agent": {
            "iteration": state.coordinator_agent.iteration + 1,
            "final_answer": final_answer,
            # "plan": plan,
            "next_agent": next_agent
        },
        "answer": answer
    }

In [64]:
results = ls_client.evaluate(
    lambda x: coordinator_agent(
        State(
            messages=x["input"]["messages"],
            coordinator_agent=CoordinatorAgentProperties(
                iteration=0,
                final_answer=False,
                plan=[],
                next_agent=""
            ),
            answer=""
        )
    ),
    data="coordinator-delegation-evaluation-2",
    evaluators=[
        evaluate_coordinator_delegation
    ],
    experiment_prefix="coordinator-delegation",
    max_concurrency=10,
    num_repetitions=5
)

View the evaluation results for experiment: 'coordinator-delegation-a09ab963' at:
https://smith.langchain.com/o/b99d0ed7-3848-4b7d-a09d-6fa565e05a8b/datasets/368398f5-4900-4da2-91da-32618bf864fb/compare?selectedSessions=b5802505-3c1b-4380-bef5-301aa0988c05




0it [00:00, ?it/s]Error running target function: 'messages'
Traceback (most recent call last):
  File "/Users/tk/Projects/chatbot_poc_v2/.venv/lib/python3.12/site-packages/langsmith/evaluation/_runner.py", line 1976, in _forward
    fn(*args, langsmith_extra=langsmith_extra)
  File "/var/folders/x9/1y3l4mfd6hbgv0p0j9p7h76m0000gn/T/ipykernel_9153/4062312481.py", line 4, in <lambda>
    messages=x["input"]["messages"],
             ~~~~~~~~~~^^^^^^^^^^^^
KeyError: 'messages'
Error running target function: 'messages'
Traceback (most recent call last):
  File "/Users/tk/Projects/chatbot_poc_v2/.venv/lib/python3.12/site-packages/langsmith/evaluation/_runner.py", line 1976, in _forward
    fn(*args, langsmith_extra=langsmith_extra)
  File "/var/folders/x9/1y3l4mfd6hbgv0p0j9p7h76m0000gn/T/ipykernel_9153/4062312481.py", line 4, in <lambda>
    messages=x["input"]["messages"],
             ~~~~~~~~~~^^^^^^^^^^^^
KeyError: 'messages'
Error running target function: 'messages'
Traceback (most rece

In [65]:
results = ls_client.evaluate(
    lambda x: coordinator_agent(
        State(
            messages=x["input"]["messages"],
            coordinator_agent=CoordinatorAgentProperties(
                iteration=0,
                final_answer=False,
                plan=[],
                next_agent=""
            ),
            answer=""
        )
    ),
    data="coordinator-delegation-evaluation-2",
    evaluators=[
        evaluate_coordinator_delegation
    ],
    experiment_prefix="coordinator-delegation",
    max_concurrency=10,
    num_repetitions=10
)

View the evaluation results for experiment: 'coordinator-delegation-24179a1c' at:
https://smith.langchain.com/o/b99d0ed7-3848-4b7d-a09d-6fa565e05a8b/datasets/368398f5-4900-4da2-91da-32618bf864fb/compare?selectedSessions=861c79b1-5508-4d9c-83bb-98d271f61621




0it [00:00, ?it/s]Error running target function: 'messages'
Traceback (most recent call last):
  File "/Users/tk/Projects/chatbot_poc_v2/.venv/lib/python3.12/site-packages/langsmith/evaluation/_runner.py", line 1976, in _forward
    fn(*args, langsmith_extra=langsmith_extra)
  File "/var/folders/x9/1y3l4mfd6hbgv0p0j9p7h76m0000gn/T/ipykernel_9153/3609917973.py", line 4, in <lambda>
    messages=x["input"]["messages"],
             ~~~~~~~~~~^^^^^^^^^^^^
KeyError: 'messages'
Error running target function: 'messages'
Traceback (most recent call last):
  File "/Users/tk/Projects/chatbot_poc_v2/.venv/lib/python3.12/site-packages/langsmith/evaluation/_runner.py", line 1976, in _forward
    fn(*args, langsmith_extra=langsmith_extra)
  File "/var/folders/x9/1y3l4mfd6hbgv0p0j9p7h76m0000gn/T/ipykernel_9153/3609917973.py", line 4, in <lambda>
    messages=x["input"]["messages"],
             ~~~~~~~~~~^^^^^^^^^^^^
KeyError: 'messages'
Error running target function: 'messages'
Traceback (most rece

### Coordinator Agent V6

In [66]:
class Delegation(BaseModel):
    agent: str = Field(description="The agent to delegate the task to.")
    task: str = Field(description="The task to be performed by the agent.")

class Plan(BaseModel):
    next_agent: str = Field(description="The next agent to invoke")
    # plan: List[Delegation] = Field(description="A list of delegations to agents with tasks to be performed in sequence.")

class FinalAgentResponse(BaseModel):

    answer: str = Field(description="Answer to the question")

class CoordinatorAgentProperties(BaseModel):
    iteration: int = 0
    final_answer: bool = False
    # plan: List[Delegation] = []
    next_agent: str = ""
    
class State(BaseModel):
    messages: Annotated[List[Any], add] = []
    coordinator_agent: CoordinatorAgentProperties = CoordinatorAgentProperties()
    answer: str = ""

In [67]:
@traceable(
    name="coordinator_agent",
    run_type="llm",
    metadata={
        "ls_provider": "openai",
        "ls_model_name": "gpt-5.4-mini"
    }
)
def coordinator_agent(state) -> dict:
    
    prompt_template = """You are a Coordinator Agent as part of a shopping assistant.

## Instructions

- Your role is to delegate work to worker agents in order to solve user queries.
- You should output the next agent to invoke.
- Once an agent finishes its task, you will be handed the control back, you should then review the conversation history and output the next delegation via the `Plan` tool.
- If you do not need to delegate work to any agent, output the final answer via the `FinalAgentResponse` tool.
- Do not route to any agent if the user's query needs clarification or is irelevant. Do it yourself.
- Only route to the `warehouse_manager_agent` if the user has confirmed that they want to reserve an item and it was already successfully added to the shopping cart.
- Only route to the `shopping_cart_agent` if the user specifically asks about their shopping cart or wants to add or remove items from the shopping cart.
- Do not delegate work to the same agent twice in a row.

## Routing rules

Match the user's latest message to one of these patterns:

1. PRODUCT QUESTION — asks about products, specs, reviews, availability, suggestions.
    Examples: "can I get some earphones", "show me laptops", "what watches do you have", "tell me about X", "is X any good", "I'd like to see some X".
    → Route ONLY to `product_qna_agent`. When it returns, emit `FinalAgentResponse`. Do NOT chain to cart or warehouse.

2. EXPLICIT CART INSTRUCTION — uses cart vocabulary.
    Examples: "add to my cart", "remove from my cart", "what's in my cart", "put X in my cart", "delete X from cart".
    → Route to `shopping_cart_agent`. (If items must be discovered first, route to `product_qna_agent` first.)

3. EXPLICIT RESERVATION INSTRUCTION — uses warehouse/reservation vocabulary.
    Examples: "reserve these", "hold these for me", "check warehouse stock for X".
    → Route to `warehouse_manager_agent` ONLY AFTER items are confirmed in the cart.

Default rule: if the user's message does NOT contain explicit cart or warehouse vocabulary, do NOT delegate to those agents. Phrases like "can I get", "I want",
"I'd like", "show me", "I need" are PRODUCT QUESTIONS, not purchase instructions. The user must explicitly say "add", "cart", "reserve", "buy", or "order" before those agents are in scope.

## Available Agents

- product_qna_agent: The user is asking a question about a product. This can be a question about available products, their specifications, user reviews etc.
- shopping_cart_agent: The user is asking to add or remove items from the shopping cart or questions about the current shopping cart.
- warehouse_manager_agent: The user is asking to reserve items from the warehouses or about availability of the items in warehouses.
"""

    template = Template(prompt_template)

    prompt = template.render()

    llm = ChatOpenAI(
        model="gpt-5.4-mini",
        reasoning_effort="low",
        use_responses_api=True
    )
    llm_with_tools = llm.bind_tools(
        [FinalAgentResponse, Plan],
        tool_choice="required"
    )

    response = llm_with_tools.invoke(
        [
            SystemMessage(content=prompt),
            *state.messages
        ]
    )

    final_answer = False
    answer = ""
    # plan = []
    next_agent = ""

    def sanitise_response(response):

        for tool_call in response.tool_calls:
            if tool_call.get("name") == "FinalAgentResponse":
                answer = tool_call.get("args").get("answer")

        return AIMessage(content=answer)

    if len(response.tool_calls) > 0:
        if response.tool_calls[0].get("name") == "Plan":
            # plan = response.tool_calls[0].get("args").get("plan")
            next_agent = response.tool_calls[0].get("args").get("next_agent")
            response = None
        else:
            for tool_call in response.tool_calls:
                if tool_call.get("name") == "FinalAgentResponse":
                    final_answer = True
                    answer = tool_call.get("args").get("answer")

                    response = sanitise_response(response)

    return {
        "messages": [response] if response else [],
        "coordinator_agent": {
            "iteration": state.coordinator_agent.iteration + 1,
            "final_answer": final_answer,
            # "plan": plan,
            "next_agent": next_agent
        },
        "answer": answer
    }

In [68]:
results = ls_client.evaluate(
    lambda x: coordinator_agent(
        State(
            messages=x["input"]["messages"],
            coordinator_agent=CoordinatorAgentProperties(
                iteration=0,
                final_answer=False,
                plan=[],
                next_agent=""
            ),
            answer=""
        )
    ),
    data="coordinator-delegation-evaluation-2",
    evaluators=[
        evaluate_coordinator_delegation
    ],
    experiment_prefix="coordinator-delegation",
    max_concurrency=10,
    num_repetitions=10
)

View the evaluation results for experiment: 'coordinator-delegation-7c34f61b' at:
https://smith.langchain.com/o/b99d0ed7-3848-4b7d-a09d-6fa565e05a8b/datasets/368398f5-4900-4da2-91da-32618bf864fb/compare?selectedSessions=f8299496-8bb1-4eec-9b01-6e4f9da841f3




0it [00:00, ?it/s]Error running target function: 'messages'
Traceback (most recent call last):
  File "/Users/tk/Projects/chatbot_poc_v2/.venv/lib/python3.12/site-packages/langsmith/evaluation/_runner.py", line 1976, in _forward
    fn(*args, langsmith_extra=langsmith_extra)
  File "/var/folders/x9/1y3l4mfd6hbgv0p0j9p7h76m0000gn/T/ipykernel_9153/3609917973.py", line 4, in <lambda>
    messages=x["input"]["messages"],
             ~~~~~~~~~~^^^^^^^^^^^^
KeyError: 'messages'
Error running target function: 'messages'
Traceback (most recent call last):
  File "/Users/tk/Projects/chatbot_poc_v2/.venv/lib/python3.12/site-packages/langsmith/evaluation/_runner.py", line 1976, in _forward
    fn(*args, langsmith_extra=langsmith_extra)
  File "/var/folders/x9/1y3l4mfd6hbgv0p0j9p7h76m0000gn/T/ipykernel_9153/3609917973.py", line 4, in <lambda>
    messages=x["input"]["messages"],
             ~~~~~~~~~~^^^^^^^^^^^^
KeyError: 'messages'
Error running target function: 'messages'
Traceback (most rece

### Coordinator Agent V7

In [69]:
class Delegation(BaseModel):
    agent: str = Field(description="The agent to delegate the task to.")
    task: str = Field(description="The task to be performed by the agent.")

class Plan(BaseModel):
    next_agent: str = Field(description="The next agent to invoke")
    # plan: List[Delegation] = Field(description="A list of delegations to agents with tasks to be performed in sequence.")

class FinalAgentResponse(BaseModel):

    answer: str = Field(description="Answer to the question")

class CoordinatorAgentProperties(BaseModel):
    iteration: int = 0
    final_answer: bool = False
    # plan: List[Delegation] = []
    next_agent: str = ""
    
class State(BaseModel):
    messages: Annotated[List[Any], add] = []
    coordinator_agent: CoordinatorAgentProperties = CoordinatorAgentProperties()
    answer: str = ""

In [70]:
@traceable(
    name="coordinator_agent",
    run_type="llm",
    metadata={
        "ls_provider": "openai",
        "ls_model_name": "gpt-5.4-mini"
    }
)
def coordinator_agent(state) -> dict:
    
    prompt_template = """You are a Coordinator Agent as part of a shopping assistant.

## Instructions

- Your role is to delegate work to worker agents in order to solve user queries.
- You should output the next agent to invoke.
- Once an agent finishes its task, you will be handed the control back, you should then review the conversation history and output the next delegation via the `Plan` tool.
- If you do not need to delegate work to any agent, output the final answer via the `FinalAgentResponse` tool.
- Do not route to any agent if the user's query needs clarification or is irelevant. Do it yourself.
- Only route to the `warehouse_manager_agent` if the user has confirmed that they want to reserve an item and it was already successfully added to the shopping cart.
- Only route to the `shopping_cart_agent` if the user specifically asks about their shopping cart or wants to add or remove items from the shopping cart.
- Do not delegate work to the same agent twice in a row.

## Routing rules

Match the user's latest message to one of these patterns:

1. PRODUCT QUESTION — asks about products, specs, reviews, availability, suggestions.
    Examples: "can I get some earphones", "show me laptops", "what watches do you have", "tell me about X", "is X any good", "I'd like to see some X".
    → Route ONLY to `product_qna_agent`. When it returns, emit `FinalAgentResponse`. Do NOT chain to cart or warehouse.

2. EXPLICIT CART INSTRUCTION — uses cart vocabulary.
    Examples: "add to my cart", "remove from my cart", "what's in my cart", "put X in my cart", "delete X from cart".
    → Route to `shopping_cart_agent`. (If items must be discovered first, route to `product_qna_agent` first.)

3. EXPLICIT RESERVATION INSTRUCTION — uses warehouse/reservation vocabulary.
    Examples: "reserve these", "hold these for me", "check warehouse stock for X".
    → Route to `warehouse_manager_agent` ONLY AFTER items are confirmed in the cart.

Default rule: if the user's message does NOT contain explicit cart or warehouse vocabulary, do NOT delegate to those agents. Phrases like "can I get", "I want",
"I'd like", "show me", "I need" are PRODUCT QUESTIONS, not purchase instructions. The user must explicitly say "add", "cart", "reserve", "buy", or "order" before those agents are in scope.

## Available Agents

- product_qna_agent: The user is asking a question about a product. This can be a question about available products, their specifications, user reviews etc.
- shopping_cart_agent: The user is asking to add or remove items from the shopping cart or questions about the current shopping cart.
- warehouse_manager_agent: The user is asking to reserve items from the warehouses or about availability of the items in warehouses.
"""

    template = Template(prompt_template)

    prompt = template.render()

    llm = ChatOpenAI(
        model="gpt-5.4-mini",
        reasoning_effort="medium",
        use_responses_api=True
    )
    llm_with_tools = llm.bind_tools(
        [FinalAgentResponse, Plan],
        tool_choice="required"
    )

    response = llm_with_tools.invoke(
        [
            SystemMessage(content=prompt),
            *state.messages
        ]
    )

    final_answer = False
    answer = ""
    # plan = []
    next_agent = ""

    def sanitise_response(response):

        for tool_call in response.tool_calls:
            if tool_call.get("name") == "FinalAgentResponse":
                answer = tool_call.get("args").get("answer")

        return AIMessage(content=answer)

    if len(response.tool_calls) > 0:
        if response.tool_calls[0].get("name") == "Plan":
            # plan = response.tool_calls[0].get("args").get("plan")
            next_agent = response.tool_calls[0].get("args").get("next_agent")
            response = None
        else:
            for tool_call in response.tool_calls:
                if tool_call.get("name") == "FinalAgentResponse":
                    final_answer = True
                    answer = tool_call.get("args").get("answer")

                    response = sanitise_response(response)

    return {
        "messages": [response] if response else [],
        "coordinator_agent": {
            "iteration": state.coordinator_agent.iteration + 1,
            "final_answer": final_answer,
            # "plan": plan,
            "next_agent": next_agent
        },
        "answer": answer
    }

In [71]:
results = ls_client.evaluate(
    lambda x: coordinator_agent(
        State(
            messages=x["input"]["messages"],
            coordinator_agent=CoordinatorAgentProperties(
                iteration=0,
                final_answer=False,
                plan=[],
                next_agent=""
            ),
            answer=""
        )
    ),
    data="coordinator-delegation-evaluation-2",
    evaluators=[
        evaluate_coordinator_delegation
    ],
    experiment_prefix="coordinator-delegation",
    max_concurrency=10,
    num_repetitions=10
)

View the evaluation results for experiment: 'coordinator-delegation-90d4f618' at:
https://smith.langchain.com/o/b99d0ed7-3848-4b7d-a09d-6fa565e05a8b/datasets/368398f5-4900-4da2-91da-32618bf864fb/compare?selectedSessions=3d36b255-5b71-4c02-ad9f-323dcdc75ef0




0it [00:00, ?it/s]Error running target function: 'messages'
Traceback (most recent call last):
  File "/Users/tk/Projects/chatbot_poc_v2/.venv/lib/python3.12/site-packages/langsmith/evaluation/_runner.py", line 1976, in _forward
    fn(*args, langsmith_extra=langsmith_extra)
  File "/var/folders/x9/1y3l4mfd6hbgv0p0j9p7h76m0000gn/T/ipykernel_9153/3609917973.py", line 4, in <lambda>
    messages=x["input"]["messages"],
             ~~~~~~~~~~^^^^^^^^^^^^
KeyError: 'messages'
Error running target function: 'messages'
Traceback (most recent call last):
  File "/Users/tk/Projects/chatbot_poc_v2/.venv/lib/python3.12/site-packages/langsmith/evaluation/_runner.py", line 1976, in _forward
    fn(*args, langsmith_extra=langsmith_extra)
  File "/var/folders/x9/1y3l4mfd6hbgv0p0j9p7h76m0000gn/T/ipykernel_9153/3609917973.py", line 4, in <lambda>
    messages=x["input"]["messages"],
             ~~~~~~~~~~^^^^^^^^^^^^
KeyError: 'messages'
Error running target function: 'messages'
Traceback (most rece

### RAG Pipeline

In [ ]:
qdrant_client = QdrantClient(url="http://localhost:6333")

def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )

    return response.data[0].embedding


def retrieve_data(query, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01",
        query=query_embedding,
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocessed_description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }

def process_context(context):

    formatted_context = ""

    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"

    return formatted_context


def build_prompt(preprocessed_context, question):

    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
{preprocessed_context}

Question:
{question}    
"""

    return prompt

def generate_answer(prompt):

    response = openai.chat.completions.create(
        model="gpt-5.4-nano",
        messages=[
            {"role": "system", "content": prompt}
        ],
        reasoning_effort="none"
    )

    return response.choices[0].message.content


def rag_pipeline(question, top_k=5):

    retrieved_context = retrieve_data(question, k=top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)

    final_answer = {
        "answer": answer,
        "question": question,
        "retrieved_context_ids": retrieved_context["retrieved_context_ids"],
        "retrieved_context": retrieved_context["retrieved_context"]
    }

    return final_answer

In [ ]:
rag_pipeline("Can I get a charger?")

{'answer': 'Yes. We have:\n\n1) iPhone Charger 6ft 3Pack (Apple MFi Certified Lightning Cables) – fast charging up to 3A with high-speed data sync, reinforced durability, and compatibility with many iPhone and iPad models.\n\n2) USB-C to USB-C Cable (INIU) – 6.6ft, 100W PD 5A fast charging for USB-C devices like phones, tablets, and many laptops.\n\n3) Replacement Notebook Charger (White-60W) – compatible with notebooks; it’s the second generation of the power adapter (double-check your Mac notebook model before buying).\n\nWhich device are you charging (iPhone Lightning, USB-C laptop/phone, or a notebook)?',
 'question': 'Can I get a charger?',
 'retrieved_context_ids': ['B0BBVJJRHD',
  'B0BGH3H1WM',
  'B0BN1CMWCP',
  'B0C9QZS95R',
  'B0BXC72RLD'],
 'retrieved_context': ['iPhone Charger 6ft 3Pack, Apple MFi Certified Lightning Cable Fast Charging Long iPhone Charger Cord High Speed Data Sync Cable Compatible iPhone 14 13 12 11 Pro Max XS XR X 8 7 6S 6 Plus SE 5S, iPad 【iPhone Cable Fa

### RAGAS Metrics

In [ ]:
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall, Faithfulness, ResponseRelevancy

/var/folders/x9/1y3l4mfd6hbgv0p0j9p7h76m0000gn/T/ipykernel_85215/3756680326.py:2: DeprecationWarning: Importing IDBasedContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import IDBasedContextPrecision
  from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall, Faithfulness, ResponseRelevancy
/var/folders/x9/1y3l4mfd6hbgv0p0j9p7h76m0000gn/T/ipykernel_85215/3756680326.py:2: DeprecationWarning: Importing IDBasedContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import IDBasedContextRecall
  from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall, Faithfulness, ResponseRelevancy
/var/folders/x9/1y3l4mfd6hbgv0p0j9p7h76m0000gn/T/ipykernel_85215/3756680326.py:2: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is depre

In [ ]:
ragas_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-5.4-mini"))
ragas_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))

/var/folders/x9/1y3l4mfd6hbgv0p0j9p7h76m0000gn/T/ipykernel_85215/840510326.py:1: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-5.4-mini"))
/var/folders/x9/1y3l4mfd6hbgv0p0j9p7h76m0000gn/T/ipykernel_85215/840510326.py:2: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))


In [ ]:
reference_input

In [ ]:
reference_output

{'ground_truth': 'Among these options, the Wekily earbuds and the Jesebang earbuds both offer up to 40 hours total playtime with their charging cases. The pamu earbuds offer up to 30 hours total battery life. If battery life is your main priority, the Wekily and Jesebang models are the strongest choices here.',
 'reference_context_ids': ['B0BRV544MV', 'B09X9838WY', 'B09TFM1SFQ'],
 'reference_descriptions': ['Wekily Bluetooth 5.3 Headphones, Wireless Earbuds with 40H Playtimes Charge Case, Deep Bass, IPX7 Waterproof Running Earphones with 4-Mic Clear Call, LED Display, in Ear Headphones for Work/Gym Excellent Sound Quality: Wireless earbuds adopte graphene drivers and a polymer composite diaphragm carrying an AAC/SBC audio decoder, resulting in up to 43% bass enhancement. Audio technology is losslessly transmitted for a clearer stereo sound from the headphones. 40Hrs Cycle Playtimes: The 400mAh charging case has a playtime of 35 hours and the wireless headphones have a single playtime o

In [ ]:
result = rag_pipeline(reference_input["question"])

In [ ]:
result

{'answer': 'Among the wireless earbuds available, the longest total battery life is the Jesebang Wireless Earbud (B09X9838WY), with up to 40 hours of playtime (8 hours per single charge plus the charging case).',
 'question': 'Which wireless earbuds you have offer the longest battery life?',
 'retrieved_context_ids': ['B0BRV544MV',
  'B09X9838WY',
  'B0CFHWF326',
  'B09TFM1SFQ',
  'B0CH8DRD6K'],
 'retrieved_context': ['Wekily Bluetooth 5.3 Headphones, Wireless Earbuds with 40H Playtimes Charge Case, Deep Bass, IPX7 Waterproof Running Earphones with 4-Mic Clear Call, LED Display, in Ear Headphones for Work/Gym Excellent Sound Quality: Wireless earbuds adopte graphene drivers and a polymer composite diaphragm carrying an AAC/SBC audio decoder, resulting in up to 43% bass enhancement. Audio technology is losslessly transmitted for a clearer stereo sound from the headphones. 40Hrs Cycle Playtimes: The 400mAh charging case has a playtime of 35 hours and the wireless headphones have a single

In [ ]:
async def ragas_context_precision_id_based(run, example):

    sample = SingleTurnSample(
        retrieved_context_ids=run["retrieved_context_ids"],
        reference_context_ids=example["reference_context_ids"]
    )

    scorer = IDBasedContextPrecision()

    return await scorer.single_turn_ascore(sample)

In [ ]:
await ragas_context_precision_id_based(result, reference_output)

0.6

In [ ]:
async def ragas_context_recall_id_based(run, example):

    sample = SingleTurnSample(
        retrieved_context_ids=run["retrieved_context_ids"],
        reference_context_ids=example["reference_context_ids"]
    )

    scorer = IDBasedContextRecall()

    return await scorer.single_turn_ascore(sample)

In [ ]:
await ragas_context_recall_id_based(result, reference_output)

1.0

In [ ]:
async def ragas_faithfulness(run, example):

    sample = SingleTurnSample(
            user_input=run["question"],
            response=run["answer"],
            retrieved_contexts=run["retrieved_context"]
        )

    scorer = Faithfulness(llm=ragas_llm)
    
    return await scorer.single_turn_ascore(sample)

In [ ]:
await ragas_faithfulness(result, reference_output)

0.6666666666666666

In [ ]:
async def ragas_relevancy(run, example):

    sample = SingleTurnSample(
        user_input=run["question"],
        response=run["answer"],
        retrieved_contexts=run["retrieved_context"]
    )

    scorer = ResponseRelevancy(llm=ragas_llm, embeddings=ragas_embeddings)

    return await scorer.single_turn_ascore(sample)

In [ ]:
await ragas_relevancy(result, reference_output)

np.float64(0.9321365425987805)